# Deep Q-Network (DQN) Teorisi

Bu notebook DQN algoritmasını teorik olarak anlatır ve kavramları Orbit Wars kod tabanındaki `rl/dqn.py`, `rl/dqn_train.py` ve `model_q.py` yapılarıyla ilişkilendirir.

Odak noktaları:
- Markov karar süreci ve aksiyon-değer fonksiyonu
- Bellman optimalite denklemi
- Q-learning'den DQN'e geçiş
- Experience replay ve target network
- Double DQN hedefi
- Orbit Wars'taki faktörize/branching aksiyon uzayı

## 0. Sembol Sözlüğü

| Sembol | Anlam |
| --- | --- |
| $S_t$ veya $s_t$ | $t$ anındaki durum |
| $A_t$ veya $a_t$ | $t$ anında seçilen aksiyon |
| $R_{t+1}$ veya $r_{t+1}$ | $A_t$ aksiyonundan sonra gelen ödül |
| $S_{t+1}$ | Bir sonraki durum |
| $\gamma$ | Gelecek ödül iskonto katsayısı |
| $\alpha$ | Tabular Q-learning'deki öğrenme oranı |
| $Q(s,a)$ | Durum-aksiyon değer tahmini |
| $y$ | TD target; tahminin yaklaşması istenen hedef |
| $\delta_t$ | TD error; target ile mevcut tahmin arasındaki fark |
| $\theta$ | Online Q ağının parametreleri |
| $\theta^-$ | Target Q ağının parametreleri |
| $d$ | Episode bitti mi göstergesi; bittiyse bootstrap kapatılır |

Bu notebookta büyük harfli $S_t,A_t,R_{t+1}$ notasyonu ile küçük harfli $s_t,a_t,r_{t+1}$ notasyonu aynı kavramları ifade eder. Büyük harf rastgele değişkeni, küçük harf gerçekleşmiş örneği vurgular.

## 1. Problem Formülasyonu

Pekiştirmeli öğrenmede ajan, bir Markov karar sürecinde (MDP) hareket eder:

$$\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$$

- $s \in \mathcal{S}$: durum
- $a \in \mathcal{A}$: aksiyon
- $P(s' \mid s,a)$: geçiş dinamiği
- $R(s,a,s')$: ödül
- $\gamma \in [0,1)$: gelecekteki ödüllerin iskonto katsayısı

Amaç, beklenen iskontolu toplam ödülü maksimize eden bir politika bulmaktır:

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

DQN doğrudan politikayı öğrenmek yerine, her durum-aksiyon çifti için uzun vadeli değeri tahmin eden $Q(s,a)$ fonksiyonunu öğrenir.

## 2. Aksiyon-Değer Fonksiyonu

Bir politika $\pi$ altında aksiyon-değer fonksiyonu:

$$Q^{\pi}(s,a) = \mathbb{E}_{\pi}\left[G_t \mid S_t=s, A_t=a\right]$$

Optimal aksiyon-değer fonksiyonu:

$$Q^*(s,a) = \max_{\pi} Q^{\pi}(s,a)$$

Optimal politika $Q^*$ üzerinden greedy seçilebilir:

$$\pi^*(s) = \arg\max_a Q^*(s,a)$$

Bu yüzden DQN'in temel hedefi şudur: bir sinir ağı $Q_\theta(s,a)$ ile $Q^*(s,a)$'ya yaklaşmak.

## 3. Bellman Optimalite Denklemi

$Q^*$ şu sabit nokta denklemini sağlar:

$$Q^*(s,a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s',a') \mid s,a\right]$$

Tabular Q-learning güncellemesi:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left(y - Q(s,a)\right)$$

$$y = r + \gamma (1-d) \max_{a'} Q(s',a')$$

Burada $d$ episode bitti mi bilgisidir. Episode bitmişse bootstrap terimi sıfırlanır.

### 3.1. Q-Learning Güncellemesini Parçalara Ayırma

Görseldeki klasik Q-learning güncellemesi şu formdadır:

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t)+\alpha\left[R_{t+1}+\gamma\max_a Q(S_{t+1},a)-Q(S_t,A_t)\right]$$

Bu ifade birkaç parçadan oluşur:

| Parça | Formül | Anlam |
| --- | --- | --- |
| Eski Q tahmini | $Q(S_t,A_t)$ | Mevcut durumda seçilen aksiyon için eski değer tahmini |
| Immediate reward | $R_{t+1}$ | Aksiyondan hemen sonra alınan ödül |
| Discounted next estimate | $\gamma\max_a Q(S_{t+1},a)$ | Bir sonraki durumdaki en iyi aksiyonun iskontolu değeri |
| TD target | $R_{t+1}+\gamma\max_aQ(S_{t+1},a)$ | Q tahmininin yaklaşmasını istediğimiz hedef |
| TD error | $R_{t+1}+\gamma\max_aQ(S_{t+1},a)-Q(S_t,A_t)$ | Hedef ile mevcut tahmin arasındaki fark |
| TD error sembolü | $\delta_t$ | Aynı TD error'ın kısa gösterimi; update formülünde hata sinyali olarak kullanılır |
| Learning rate | $\alpha$ | TD error'ın eski tahmine ne kadar uygulanacağı |

Kısa biçimde:

$$Q_{new}=Q_{old}+\alpha\,\delta_t$$

$$\delta_t=\underbrace{R_{t+1}+\gamma\max_a Q(S_{t+1},a)}_{TD\ target}-\underbrace{Q(S_t,A_t)}_{current\ estimate}$$

Tabular Q-learning'de $Q$ tablosunun tek hücresi doğrudan güncellenir. DQN'de ise bu doğrudan atama yapılmaz; TD error'dan bir loss üretilir ve sinir ağının ağırlıkları gradient descent ile güncellenir.

## 4. DQN Kaybı

DQN, tabular tablo yerine parametreli bir ağ kullanır. Bir replay batch'i için hedef:

$$y_i = r_i + \gamma (1-d_i) \max_{a'} Q_{\theta^-}(s'_i,a')$$

Online tahmin:

$$\hat{q}_i = Q_{\theta}(s_i,a_i)$$

Kayıp çoğunlukla MSE veya Huber kaybıdır:

$$L(\theta) = \frac{1}{B}\sum_i \ell\left(\hat{q}_i - y_i\right)$$

Görseldeki DQN notasyonuyla aynı hedef genellikle şöyle yazılır:

$$y_j=r_j+\gamma\max_{a'}\hat{Q}(\phi_{j+1},a';\theta^-)$$

Burada $\phi_j$ ağın girdisi olan durum temsili, $\theta$ online ağ parametreleri, $\theta^-$ target ağ parametreleridir. Q-loss'un temel hatası:

$$y_j-Q(\phi_j,a_j;\theta)$$

Yani DQN, tabular formdaki $Q_{old}+\alpha\delta$ güncellemesini doğrudan Q tablosuna yazmaz. Bunun yerine şu farkı küçültür:

$$\delta_j=y_j-Q_\theta(s_j,a_j)$$

MSE kullanılırsa loss:

$$L(\theta)=\frac{1}{B}\sum_j\left(y_j-Q_\theta(s_j,a_j)\right)^2$$

Huber kullanılırsa aynı TD error küçükken karesel, büyükken yaklaşık doğrusal cezalandırılır. Bu yüzden outlier TD hatalarında eğitim daha stabil olur.

Orbit Wars kodunda `rl/dqn.py::dqn_update` bu yapıyı uygular. Varsayılan kayıp `smooth_l1_loss`, yani Huber kaybıdır. Huber, büyük TD hatalarında MSE'ye göre daha dayanıklıdır.

In [ ]:
import torch
import torch.nn.functional as F

def scalar_dqn_loss(q_taken, reward, done, max_q_next, gamma=0.99, huber_delta=1.0):
    """DQN kaybının minimal, tek-branch gösterimi."""
    target = reward + gamma * (1.0 - done) * max_q_next
    td_error = target - q_taken
    if huber_delta > 0:
        loss = F.smooth_l1_loss(q_taken, target.detach(), beta=huber_delta)
    else:
        loss = F.mse_loss(q_taken, target.detach())
    return {"target": target, "td_error": td_error, "loss": loss}

q_taken = torch.tensor([1.2, 0.4, -0.1])
reward = torch.tensor([1.0, 0.0, -1.0])
done = torch.tensor([0.0, 0.0, 1.0])
max_q_next = torch.tensor([0.8, 1.1, 4.0])

scalar_dqn_loss(q_taken, reward, done, max_q_next)

## 4.1. Minimal DQN Neural Network

DQN ağının görevi bir durumdan her ayrık aksiyon için $Q_	heta(s,a)$ tahmini üretmektir. Eğitimde sadece seçilmiş aksiyonun Q değeri alınır, TD target ile karşılaştırılır ve loss geriye yayılır.

Orbit Wars'taki gerçek model `OrbitWarsQNet` branch başlıkları kullanır; aşağıdaki minimal ağ ise aynı fikri tek ayrık aksiyon uzayında gösterir.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class MinimalDQN(nn.Module):
    """DQN: state -> Q(s, a_0 ... a_n). Son katmanda softmax yoktur."""
    def __init__(self, obs_dim=16, hidden_dim=128, n_actions=6):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.q_head = nn.Linear(hidden_dim, n_actions)

    def forward(self, obs):
        h = self.encoder(obs)
        return self.q_head(h)

B, obs_dim, n_actions = 4, 16, 6
online_net = MinimalDQN(obs_dim, n_actions=n_actions)
target_net = MinimalDQN(obs_dim, n_actions=n_actions)
target_net.load_state_dict(online_net.state_dict())

obs = torch.randn(B, obs_dim)
next_obs = torch.randn(B, obs_dim)
actions = torch.tensor([0, 2, 4, 1])
rewards = torch.tensor([1.0, 0.0, -0.2, 0.5])
dones = torch.tensor([0.0, 0.0, 1.0, 0.0])
gamma = 0.99

q_all = online_net(obs)                              # (B, n_actions)
q_taken = q_all.gather(1, actions[:, None]).squeeze(1)
with torch.no_grad():
    max_q_next = target_net(next_obs).max(dim=1).values
    td_target = rewards + gamma * (1.0 - dones) * max_q_next

td_error = td_target - q_taken
loss = F.smooth_l1_loss(q_taken, td_target)

print("q_all shape:", q_all.shape)
print("q_taken:", q_taken)
print("td_target:", td_target)
print("td_error:", td_error)
print("loss:", float(loss.detach()))

## 4.2. Minimal DQN Update Algoritması

DQN update algoritması şu sırayı izler:

1. Online ağ ile $Q_	heta(s,a)$ hesapla.
2. Target ağ ile $r+\gamma\max_{a'}Q_{	heta^-}(s',a')$ hedefini kur.
3. TD loss'u hesapla.
4. Online ağı gradient descent ile güncelle.
5. Belirli aralıklarla target ağı online ağdan kopyala veya soft update yap.

Aşağıdaki kod tek minibatch için tam update adımını gösterir.

In [ ]:
def hard_update(target_net, online_net):
    target_net.load_state_dict(online_net.state_dict())

@torch.no_grad()
def soft_update(target_net, online_net, tau=0.05):
    for target_param, online_param in zip(target_net.parameters(), online_net.parameters()):
        target_param.mul_(1.0 - tau).add_(online_param, alpha=tau)

def dqn_train_step(online_net, target_net, optimizer, batch, gamma=0.99):
    obs, actions, rewards, next_obs, dones = batch
    q_values = online_net(obs)
    q_taken = q_values.gather(1, actions[:, None]).squeeze(1)

    with torch.no_grad():
        max_q_next = target_net(next_obs).max(dim=1).values
        td_target = rewards + gamma * (1.0 - dones) * max_q_next

    loss = F.smooth_l1_loss(q_taken, td_target)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(online_net.parameters(), max_norm=10.0)
    optimizer.step()

    return {
        "loss": float(loss.detach()),
        "td_error_mean": float((td_target - q_taken).abs().mean().detach()),
        "q_mean": float(q_taken.mean().detach()),
    }

online_net = MinimalDQN(obs_dim=16, n_actions=6)
target_net = MinimalDQN(obs_dim=16, n_actions=6)
hard_update(target_net, online_net)
optimizer = torch.optim.Adam(online_net.parameters(), lr=1e-3)

batch = (
    torch.randn(8, 16),                 # obs
    torch.randint(0, 6, (8,)),          # actions
    torch.randn(8),                     # rewards
    torch.randn(8, 16),                 # next_obs
    torch.randint(0, 2, (8,)).float(),  # dones
)
stats = dqn_train_step(online_net, target_net, optimizer, batch)
soft_update(target_net, online_net, tau=0.05)
stats

## 5. Experience Replay Neden Gerekli?

DQN off-policy bir yöntemdir. Ortamdan veri toplayan behavior policy çoğunlukla epsilon-greedy'dir; öğrenilen target policy ise Q değerlerine göre greedy politikadır. Bu ayrım replay buffer'ın teorik gerekçesidir: veri eski veya daha keşifçi bir politikadan gelmiş olsa bile Q-learning hedefi greedy optimal Q fonksiyonuna doğru güncelleme yapar.

DQN'de ajan ortamdan ardışık geçişler toplar:

$$(s_t, a_t, r_{t+1}, s_{t+1}, d_{t+1})$$

Ardışık örnekler yüksek korelasyonludur. Sinir ağını bu sırayla eğitmek optimizasyonu kararsızlaştırır. Replay buffer iki iş yapar:

1. Örnekleri karıştırarak korelasyonu azaltır.
2. Eski deneyimleri tekrar kullanarak sample efficiency'yi artırır.

Orbit Wars'ta `rl/replay.py` geçişleri saklar. `rl/dqn_train.py` self-play sırasında her environment adımında iki oyuncu perspektifinden transition ekler. Eğitim `learn_starts` kadar örnek birikene kadar başlamaz.

## 6. Target Network

DQN'in hedefi kendi tahminine bağlıdır. Aynı ağ hem $Q_\theta(s,a)$ hem de hedef $\max_{a'} Q_\theta(s',a')$ için kullanılırsa hedef sürekli hareket eder.

Bu yüzden iki ağ tutulur:

- Online network: $Q_\theta$, gradient ile güncellenir.
- Target network: $Q_{\theta^-}$, daha yavaş güncellenir.

Hedef:

$$y = r + \gamma(1-d)\max_{a'} Q_{\theta^-}(s',a')$$

Target güncelleme iki şekilde yapılabilir:

- Hard update: belirli aralıklarla $\theta^- \leftarrow \theta$
- Soft/Polyak update: $\theta^- \leftarrow \tau\theta + (1-\tau)\theta^-$

Orbit Wars kodunda ikisi de var: `hard_update_target` ve `soft_update_target`.

## 7. Double DQN

Standart DQN'de aynı target ağ hem aksiyon seçer hem de o aksiyonu değerlendirir:

$$y_{DQN}=r+\gamma Q_{\theta^-}(s',\arg\max_{a'}Q_{\theta^-}(s',a'))$$

Maksimum operatörü gürültülü Q tahminlerinde pozitif bias üretebilir. Double DQN seçim ve değerlendirmeyi ayırır:

$$a^* = \arg\max_{a'} Q_{\theta}(s',a')$$

$$y_{DDQN}=r+\gamma Q_{\theta^-}(s',a^*)$$

Orbit Wars'taki `DQNConfig.double_dqn=True` varsayılanı bunu kullanır. `online_net` greedy aksiyonu seçer, `target_net` o aksiyonun değerini ölçer.

## 8. Orbit Wars'ta Faktörize Aksiyon Uzayı

Orbit Wars aksiyonu tek bir kategorik seçim değildir. Bir hamle şu parçalardan oluşur:

- `src`: kaynak gezegen veya NO-OP
- `tgt`: hedef gezegen
- `ship_bin`: gönderilecek gemi oranı/kutusu

Tüm kombinasyonları tek devasa aksiyon uzayı yapmak pahalıdır. Bu repo Branching DQN yaklaşımına benzer bir toplam Q ayrıştırması kullanır:

$$Q_{total}(s, src, tgt, ship) = Q_{src}(s,src) + \mathbf{1}_{src\neq NOOP}\left(Q_{tgt}(s,src,tgt) + Q_{ship}(s,src,ship)\right)$$

Bunun etkisi:

- Aksiyon uzayı çarpımsal büyümek yerine branch başlıklarına ayrılır.
- Maskeler geçersiz kaynakları, padding gezegenlerini ve self-target aksiyonlarını engeller.
- Greedy seçim branch bazında yapılabilir: önce `src`, sonra seçilen `src` için `tgt` ve `ship_bin`.

In [ ]:
def q_total(q_src, q_tgt, q_ship, src, tgt, ship_bin, noop_index):
    """Orbit Wars Q ayrıştırmasının küçük tensör gösterimi."""
    base = q_src.gather(1, src[:, None]).squeeze(1)
    non_noop = (src != noop_index).float()
    src_safe = src.clamp(max=noop_index - 1)
    tgt_value = q_tgt[torch.arange(src.numel()), src_safe, tgt]
    ship_value = q_ship[torch.arange(src.numel()), src_safe, ship_bin]
    return base + non_noop * (tgt_value + ship_value)

B, P, n_bins = 2, 4, 5
q_src = torch.randn(B, P + 1)
q_tgt = torch.randn(B, P, P)
q_ship = torch.randn(B, P, n_bins)
src = torch.tensor([0, P])       # ikinci örnek NO-OP
tgt = torch.tensor([2, 1])
ship_bin = torch.tensor([3, 0])

q_total(q_src, q_tgt, q_ship, src, tgt, ship_bin, noop_index=P)

## 9. Hiperparametrelerin Teorik Rolü

| Parametre | Rol | Yanlış ayar belirtisi |
| --- | --- | --- |
| `gamma` | Gelecek ödüllere verilen önem | Çok düşükse kısa vadeli, çok yüksekse noisy hedef |
| `learning_rate` | Q fonksiyonunun güncelleme hızı | Yüksekse divergence, düşükse yavaş öğrenme |
| `batch_size` | TD hedeflerinin örnekleme varyansı | Çok küçükse gürültülü, çok büyükse yavaş |
| `replay_capacity` | Deneyim çeşitliliği | Çok küçükse unutma, çok büyükse eski politika verisi |
| `learn_starts` | İlk güncellemeden önce veri birikimi | Çok düşükse erken overfit/kararsızlık |
| `eps_decay_steps` | Keşiften sömürüye geçiş hızı | Çok hızlı düşerse lokal stratejiye kilitlenme |
| `target_sync_interval` / `target_tau` | Hedef ağın hareket hızı | Çok hızlıysa hedef oynak, çok yavaşsa stale target |

DQN off-policy olduğu için replay'de eski davranış politikasından gelen veriyi kullanabilir. Ancak self-play ortamlarında karşı oyuncu da değiştiği için veri dağılımı zamanla kayar; bu yüzden epsilon schedule, replay kapasitesi ve target update hızı birlikte düşünülmelidir.

## 10. Orbit Wars İçin Pratik Okuma Haritası

- `model_q.py`: Q ağının encoder, branch head ve greedy seçim yapısı
- `rl/dqn.py`: TD hedefi, Double DQN, Huber loss ve target update fonksiyonları
- `rl/dqn_train.py`: self-play veri toplama, epsilon schedule, replay sampling, checkpoint ve loglama
- `rl/replay.py`: transition saklama ve minibatch üretimi

DQN'i değerlendirirken sadece loss düşüşüne bakmak yeterli değildir. TD error, ortalama Q değeri, win-rate dengesi, aksiyon çeşitliliği ve replay'deki reward dağılımı birlikte izlenmelidir.